# Lesson 01: How does a machine learn a straight line?

**By the end you'll be able to**
- explain what "training a model" means, in plain English, to someone who's never coded
- build linear regression and gradient descent yourself in NumPy, and say what every line does
- read a loss curve and tell whether the learning rate is too small, too big or about right
- check gradient code numerically, the first debugging tool ML engineers reach for
- map everything you built onto scikit-learn and PyTorch, and say where it's used in finance

**You need:** Python and [lesson 00](../00-maths-toolkit/). Whenever a symbol confuses you, the *Reading maths like code* table in lesson 00, section 1, has it.

**How it works:** ▶ **Run this** = a demo, just run it. ✍️ **Your turn** = you write the code. 🔮 **Predict** = write your guess *before* running. ✅ **Check yourself** = quick questions with answers folded away. 🏭 **In industry** = how it's done for real.

Your implementation goes in `linreg.py`. Check it with `uv run pytest lessons/01-linear-regression`. Don't delete failed attempts: comment them out and note what went wrong in this lesson's README.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact

%load_ext autoreload
%autoreload 2
import linreg  # your code; autoreload picks up edits without restarting the kernel

## 1. The question

If the stock market goes up 1% today, how much does a particular stock tend to move?

Finance calls this number the stock's **beta** ($\beta$). A beta of 1.3 means the stock tends to move 1.3% for every 1% the market moves, so it amplifies the market. A beta of 0.5 means it's calmer than the market. Banks and funds use beta constantly, for example to measure how much market risk a portfolio carries, or how much to hedge.

**Beta is just the slope of a straight line drawn through a scatter plot.** That's the whole finance part. The interesting question is *how a computer finds the best line*, and the method you'll build is the same one, scaled up, that trains neural networks.

We'll start with **synthetic data**: fake data where we choose the true beta ourselves, so we can check whether the method finds it. With real data you never know the true answer, which is exactly why you test on fake data first.

In [ ]:
rng = np.random.default_rng(seed=0)

n_days = 250  # about one trading year
true_beta, true_alpha = 1.3, 0.02

market = rng.normal(loc=0.05, scale=1.0, size=n_days)  # the market's daily return, in %
noise = rng.normal(loc=0.0, scale=1.2, size=n_days)    # everything else that moves the stock
stock = true_alpha + true_beta * market + noise        # the stock's daily return, in %

market[:5], stock[:5]

✍️ **Your turn:** make a scatter plot with the market's return on the x-axis and the stock's return on the y-axis, with labelled axes. Would a straight line describe it? How spread out are the points around that line?

In [ ]:
# Your turn: scatter plot


## 2. A model is a guess with knobs

A **model** is a formula that turns an input into a prediction. The simplest one that could describe that scatter plot is a straight line:

$$\hat{y} = w x + b$$

Read it aloud: *"the prediction is w times x, plus b."*

- $x$ is the input: the market's return today.
- $\hat{y}$ ("y-hat") is the model's *prediction* of the stock's return. The hat means "estimated", to keep it apart from the real value $y$.
- $w$, the **weight**, is the slope: how much the prediction moves when $x$ moves by 1. Here it plays the part of beta.
- $b$, the **bias** or intercept, shifts the whole line up or down. (Finance calls it alpha.)

$w$ and $b$ are the model's **parameters**: its knobs. **Training a model means finding the knob settings that make its predictions as good as possible on the data you have.** That's the whole idea. The rest of this lesson is about doing it precisely, and automatically.

✍️ **Your turn:** pick three $(w, b)$ pairs by eye and draw each line over your scatter plot. Which looks best? How would you *convince* someone it's the best, rather than just saying it looks right? That's the next section.

In [ ]:
# Your turn: three hand-picked lines


## 3. Measuring "wrong": the loss function

For each day $i$, the **error** (or **residual**) is how far the prediction landed from the truth: $\hat{y}_i - y_i$. We want *one number* that sums up how wrong the line is across all the days, so that "better line" means "smaller number".

*Attempt 1: add up the errors.* This fails. A prediction 2 too high and another 2 too low add up to 0, so a terrible line can score perfectly.

*Attempt 2: add up the sizes of the errors*, ignoring the sign: $|\hat{y}_i - y_i|$. This works and is sometimes used. But it has a sharp corner at 0, which makes the slopes in section 5 awkward.

*Attempt 3: square each error, then take the average.* This is the **mean squared error** (MSE), and it's the standard choice.

**Reading the formula**, piece by piece:

$$L(w, b) = \underbrace{\frac{1}{n} \sum_{i=1}^{n}}_{\text{average over all days}} \underbrace{(\hat{y}_i - y_i)^2}_{\text{how wrong on day } i\text{, squared}}$$

In one sentence: **MSE is the average squared miss.**

Why squaring is a good idea:
- every error counts as positive, so nothing cancels out
- it's smooth, so slopes are easy to work out (section 5)
- big misses cost much more than small ones: a miss of 3 costs 9, a miss of 1 costs 1

That last point cuts both ways. A few extreme days, like a market crash, can drag the line around a lot. Keep that in mind; it comes back in the industry section.

A function like this, which turns a model's settings into "how wrong", is called a **loss function** (or cost function). Notice $L$ is written as a function of $w$ and $b$, *not* $x$ and $y$: the data is fixed, and the only things we can change are the knobs.

**Worked example.** Three days, and the line $w = 1, b = 0$:

| $x$ | $y$ | $\hat{y} = 1 \cdot x + 0$ | error $\hat{y} - y$ | error² |
|---|---|---|---|---|
| 1 | 2 | 1 | −1 | 1 |
| 2 | 2 | 2 | 0 | 0 |
| 3 | 5 | 3 | −2 | 4 |

$L = (1 + 0 + 4) / 3 = 5/3 \approx 1.67$

> **A note on shapes** (lesson 00, section 8). `linreg.py` takes a 2D input `X` with shape `(n, d)`: `n` examples (days) by `d` features (inputs). We have one feature, so `X = market.reshape(-1, 1)` has shape `(250, 1)`, and `w` is an array with one entry. The model for every row at once is $\hat{y} = X\mathbf{w} + b$: one dot product per row, plus $b$. It looks fussy for one input, but the same code will handle many inputs in section 9.

✍️ **Your turn**
1. Implement `predict` and `mse` in `linreg.py`. Run `uv run pytest lessons/01-linear-regression -k "Predict or MSE"`.
2. Compute the loss of your three hand-picked lines. Does the best-looking one have the lowest loss?

In [ ]:
X = market.reshape(-1, 1)
y = stock

# Your turn: the loss of your three lines


▶ **Run this** once `predict` and `mse` pass. Drag the sliders and try to get the MSE as low as you can by hand. The red lines are the errors; MSE is the average of their lengths squared. Notice how the long red lines dominate.

(Sliders only work when you run the notebook yourself; on GitHub you'll just see a still picture.)

In [ ]:
# ▶ Run this: fit the line by hand
def show_line(w=0.0, b=0.0):
    y_pred = linreg.predict(X, np.array([w]), b)
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.vlines(market, stock, y_pred, color="tab:red", alpha=0.35, linewidth=1, label="errors")
    ax.scatter(market, stock, s=10, alpha=0.6)
    xs = np.array([market.min(), market.max()])
    ax.plot(xs, w * xs + b, color="black", linewidth=2, label="your line")
    ax.set(xlabel="Market return (%)", ylabel="Stock return (%)",
           title=f"w = {w:.2f}, b = {b:.2f}    MSE = {linreg.mse(y_pred, stock):.3f}")
    ax.legend()
    plt.show()

interact(show_line, w=(-1.0, 3.0, 0.05), b=(-2.0, 2.0, 0.05));

### ✅ Check yourself

1. Why not just add up the errors?
2. If every prediction is off by exactly 2 (some too high, some too low), what's the MSE?
3. A line has an MSE of exactly 0. What does that tell you about the data?

<details>
<summary>Answers (try first)</summary>

1. Positive and negative errors cancel, so a terrible line could score 0.
2. 4. Every squared error is $2^2 = 4$, and the average of a lot of 4s is 4.
3. Every point lies exactly on the line. Real data with noise never does that, so an MSE of 0 on real data is usually a sign something's wrong (lesson 03).

</details>

## 4. The loss landscape

Picture a map. Every location on it is one choice of $(w, b)$, and the height at that location is the loss. **Training means finding the lowest point on the map.**

For mean squared error with a straight-line model, this landscape is always a smooth bowl with exactly one bottom. That's because the loss is built from squares of $w$ and $b$, like $(x - 3)^2 + 1$ in lesson 00, section 3, but in two directions at once. It's lucky: there's exactly one best answer and no false bottoms to get stuck in. Neural networks don't have this luxury, which is a story for lesson 09.

✍️ **Your turn**
1. Hold $b$ at 0. Compute the loss for 100 values of $w$ between −1 and 4, and plot loss against $w$. 🔮 Predict the shape, and roughly where the minimum is, before you run it.
2. Now vary both. Make a grid of $w$ and $b$ values with `np.meshgrid`, compute the loss at every point, and draw it with `plt.contour` (each ring joins points of equal loss, like height lines on a hiking map). Where's the bottom? Keep this plot; you'll draw on it in section 6.

In [ ]:
# 🔮 Prediction:

# Your turn: loss against w


In [ ]:
# Your turn: contour plot of the loss over (w, b)


In [ ]:
# ▶ Run this (after your contour plot): the same landscape in 3D
W, B = np.meshgrid(np.linspace(-1, 4, 60), np.linspace(-3, 3, 60))
loss = ((W[..., None] * market + B[..., None] - stock) ** 2).mean(axis=-1)

ax = plt.figure(figsize=(8, 6)).add_subplot(projection="3d")
ax.plot_surface(W, B, loss, cmap="viridis", alpha=0.9)
ax.set(xlabel="w (slope)", ylabel="b (intercept)", zlabel="loss (MSE)", title="The loss landscape is a bowl")
plt.show()

### ✅ Check yourself

1. On your contour plot, what do points on the same ring have in common?
2. Why does this landscape have only one lowest point?

<details>
<summary>Answers (try first)</summary>

1. They all have the same loss: different lines that are exactly as wrong as each other.
2. The loss is a sum of squares of things that change in a straight line with $w$ and $b$, which always makes a single bowl, the 2D version of $(x - 3)^2 + 1$.

</details>

## 5. Which way is downhill? The gradient

With two knobs you can search a grid, as you just did. A neural network has millions of knobs, where a grid is hopeless. Instead: **from wherever you are, work out which way is downhill, and step that way.**

You already know how to find "which way is downhill" from lesson 00: **the nudge test**. Nudge $w$ a tiny bit, see how the loss changes, and do the same for $b$. The two slopes together are the **gradient**, and minus the gradient points downhill.

The nudge test works, but it's expensive: two loss calculations per knob, so a model with a million knobs would need two million passes over the data *for every single step*. So we want a formula that gives the slopes directly, in one pass.

**The formula, in plain English first:**
- **slope for $w$** = 2 × the average of (error × input)
- **slope for $b$** = 2 × the average error

**Why that makes sense.** Suppose the line predicts too high ($\hat{y} > y$, positive error) mostly on days when the market was up a lot (big $x$). Then error × input is mostly positive, the slope for $w$ is positive, and gradient descent *lowers* $w$, making the line less steep. That's exactly what you'd do by hand. And if the line is too high on average everywhere, the slope for $b$ is positive, so $b$ goes down and the whole line shifts down. Also right.

**The same thing in notation:**

$$\frac{\partial L}{\partial w} = \frac{2}{n}\sum_{i=1}^{n} (\hat{y}_i - y_i)\,x_i \qquad\qquad \frac{\partial L}{\partial b} = \frac{2}{n}\sum_{i=1}^{n} (\hat{y}_i - y_i)$$

With several features, $\frac{\partial L}{\partial \mathbf{w}} = \frac{2}{n} X^\top (\hat{y} - y)$: the transpose turns "one error per day" back into "one slope per feature" (lesson 00, section 8).

### Optional: where the formula comes from

You don't need this to use the formula. It's here for when you want to see that nothing is pulled out of thin air. It's the chain rule (lesson 00, section 6), one step at a time.

1. For a single day, the loss is $e^2$, where the error is $e = wx + b - y$.
2. That's a chain: the knob $w$ changes $e$, and $e$ changes $e^2$.
3. Outer slope: $e^2$ changes at $2e$ per unit of $e$ (the $x^2 \to 2x$ rule).
4. Inner slope: $e = wx + b - y$ changes at $x$ per unit of $w$ (a straight line in $w$ with steepness $x$), and at $1$ per unit of $b$.
5. Rates multiply: the slope of $e^2$ with respect to $w$ is $2e \cdot x$, and with respect to $b$ it's $2e \cdot 1$.
6. The loss is the *average* over days, and the slope of an average is the average of the slopes. That gives $\frac{2}{n}\sum e_i x_i$ and $\frac{2}{n}\sum e_i$, the formulas above.

✍️ **Your turn**
1. Implement `gradients` in `linreg.py`.
2. **Gradient check.** How do you know your formula code is right? Compare it with the nudge test. The cell below borrows your `numerical_gradient` from lesson 00. Try a few random $(w, b)$; the two should agree to about six decimal places. This is how ML engineers debug gradient code, and `TestGradients` does the same.
3. 🔮 What should the gradient be at the very bottom of the bowl? Check it.

In [ ]:
import sys
sys.path.append("../00-maths-toolkit")
from toolkit import numerical_gradient

def loss_at(params):
    w, b = params
    return linreg.mse(linreg.predict(X, np.array([w]), b), y)

# Your turn: compare numerical_gradient(loss_at, [w, b]) with linreg.gradients(X, y, np.array([w]), b)


### ✅ Check yourself

1. The slope for $w$ is +3. Should $w$ go up or down to reduce the loss?
2. What does a gradient of $(0, 0)$ mean?
3. Why bother with the formula when the nudge test works?

<details>
<summary>Answers (try first)</summary>

1. Down. Positive slope means increasing $w$ increases the loss.
2. You're at a flat point: for this bowl, the bottom. No direction is downhill, so gradient descent stops moving.
3. Speed. The formula gets every slope in one pass over the data; the nudge test needs two passes per knob. Real libraries compute gradients automatically for exactly this reason (lesson 08).

</details>

## 6. Gradient descent

You did this with one knob in lesson 00, section 4. Now with two. Start from any guess, say $w = 0, b = 0$, and repeat:

$$w \leftarrow w - \alpha \frac{\partial L}{\partial w} \qquad\qquad b \leftarrow b - \alpha \frac{\partial L}{\partial b}$$

Read it aloud: *"w becomes w minus a little bit of its slope."* $\alpha$ is the **learning rate**: how big each step is. (Yes, finance uses $\alpha$ for something else; context makes it clear.) Each step here uses every day of data, so one step is one **epoch**, a full pass over the dataset.

A picture to hold on to: you're on a hillside in thick fog, trying to reach the valley floor. You can't see the valley, but you can feel the slope under your feet. So you step in the steepest downhill direction, feel the slope again, and repeat. The learning rate is your stride length.

**One step by hand**, using the three days from section 3, starting where `fit` will start, at $w = 0, b = 0$, with $\alpha = 0.1$:

| Step | Working |
|---|---|
| predictions $\hat{y}$ | 0, 0, 0 |
| errors $\hat{y} - y$ | −2, −2, −5 |
| loss | $(4 + 4 + 25)/3 = 11$ |
| slope for $w$ = 2 × average of (error × input) | $\frac{2}{3}\left((-2)(1) + (-2)(2) + (-5)(3)\right) = \frac{2}{3}(-21) = -14$ |
| slope for $b$ = 2 × average error | $\frac{2}{3}(-2 - 2 - 5) = -6$ |
| new $w$ | $0 - 0.1 \times (-14) = 1.4$ |
| new $b$ | $0 - 0.1 \times (-6) = 0.6$ |
| new loss | predictions 2.0, 3.4, 4.8 → errors 0, 1.4, −0.2 → $(0 + 1.96 + 0.04)/3 \approx 0.67$ |

Both slopes were negative (the line started far too low), so both knobs went up, and the loss fell from 11 to 0.67 in one step.

✍️ **Your turn**
1. Do the *second* step by hand (or in a cell, line by line). Did the loss go down again?
2. Implement `fit` in `linreg.py`. It should record the loss at the start of every epoch, so you can watch training progress. Run `-k Fit`. Check it against your hand-worked numbers: on the three days with `lr=0.1, epochs=3`, the history should start `[11.0, 0.667, ...]`, and the third entry should match your second step.
3. Fit the beta data with `lr=0.1, epochs=100`, and plot loss against epoch. This is a **loss curve**, and you'll look at a lot of them.
4. Record $(w, b)$ after every step (copy `fit` into the notebook, or give it an optional extra output) and draw the path on your contour plot from section 4. Does it head straight for the bottom?
5. How close did you get to the true beta of 1.3?

In [ ]:
# Your turn: the second step by hand


In [ ]:
# Your turn: fit, loss curve


In [ ]:
# Your turn: descent path on the contour plot


## 7. Breaking it on purpose: the learning rate

You'll run gradient descent for 50 epochs with five learning rates: `0.001`, `0.1`, `0.5`, `0.9` and `1.0`.

🔮 **Before running anything**, write down what you expect each loss curve to look like, and rank the five from fastest to slowest.

Then run them all and plot the five loss curves on one chart. Use a log scale on the y-axis (`plt.yscale("log")`) so big and small losses both show up. Also draw the paths for `0.9` and `1.0` on your contour plot.

In [ ]:
# 🔮 Predictions:
# 0.001:
# 0.1:
# 0.5:
# 0.9:
# 1.0:


In [ ]:
# Your turn: five loss curves on one chart


### What's going on (read after you've run it)

- **Too small** (`0.001`): every step is safe but tiny. The loss barely moves in 50 epochs.
- **Good** (`0.1`, `0.5`): the loss drops quickly and smoothly.
- **Too big, but still stable** (`0.9`): each step jumps *past* the bottom of the bowl and lands on the opposite slope. The next step jumps back. Each bounce is a little smaller than the last, so it still gets there, but more slowly than `0.5`. Bigger isn't always faster.
- **Unstable** (`1.0`): each bounce lands *higher* up the other side than where it started. The loss grows every epoch, and given long enough it overflows to `inf` and then `nan` ("not a number").

There's a sharp cliff edge between "bounces but settles" and "explodes". For this dataset it's at about 0.94, which is why `0.9` and `1.0` behave so differently even though they're close. Where the edge sits depends on how steep the bowl is: steeper bowl, lower edge. (Stretch: find it by experiment. If you take linear algebra later, it's $2 / \lambda_{\max}$, where $\lambda_{\max}$ is the largest eigenvalue of $\frac{2}{n} \tilde{X}^\top \tilde{X}$.)

Most people expect a bigger learning rate just to be faster. Write down what you predicted and what happened in the README's *What broke* section.

### ✅ Check yourself

1. Your loss curve goes up and up and ends in `nan`. What's the first thing to try?
2. Your loss curve is a smooth, very slow slope downwards. What would you try?

<details>
<summary>Answers (try first)</summary>

1. Lower the learning rate (and check your features are scaled; see section 9).
2. Raise the learning rate a bit, or train for more epochs.

</details>

## 8. Checking against the exact answer

For linear regression specifically, you don't actually *need* gradient descent. At the bottom of the bowl the slopes are zero, and you can solve "slopes = 0" directly with algebra. Add a column of ones to $X$ to handle $b$ (call the result $\tilde{X}$), and the answer is the **normal equation**:

$$\begin{pmatrix} \mathbf{w} \\ b \end{pmatrix} = (\tilde{X}^\top \tilde{X})^{-1} \tilde{X}^\top y$$

You don't need to follow every symbol. The point is that *there's a formula that jumps straight to the bottom of the bowl*.

So why learn gradient descent at all? Because almost no other model has a formula like this. Logistic regression (lesson 02) doesn't, and neural networks certainly don't. Gradient descent works for all of them. Linear regression is the one case where you can check gradient descent against an exact answer, which is why we start here.

✍️ **Your turn**
1. Implement `fit_closed_form`. Use `np.linalg.lstsq` (or `np.linalg.solve`), not `np.linalg.inv`. Inverting a matrix is slower and less accurate, and on real data it can fail completely when two columns are nearly identical. Run `-k ClosedForm`.
2. Compare four numbers: the true beta (1.3), your closed-form estimate, your gradient descent estimate, and scikit-learn's (`from sklearn.linear_model import LinearRegression`; the fitted slope is in `.coef_`). The last three should agree closely.
3. 🔮 Why doesn't *any* of them come out at exactly 1.3? Test your explanation: what happens to the gap with `n_days = 25`, and with `n_days = 25_000`?

In [ ]:
# Your turn: closed form vs gradient descent vs scikit-learn vs the truth


### ✅ Check yourself

1. With only 25 days of data, the estimate is much further from 1.3 than with 25,000 days. Why?

<details>
<summary>Answers (try first)</summary>

1. The noise. With few days, random luck in those particular days pulls the line around; with many days the luck averages out. The estimate is only as good as the data behind it, and that's true of every model you'll ever train.

</details>

## 9. More than one input, and a trap

Real models use many inputs, or **features**. Let's add the day's trading volume, measured in shares. The numbers are in the millions.

In [ ]:
volume = rng.normal(loc=5_000_000, scale=1_000_000, size=n_days)  # shares traded that day
X2 = np.column_stack([market, volume])
X2.shape

If you wrote `predict`, `gradients` and `fit` with matrix operations, they already handle two features.

✍️ **Your turn**
1. 🔮 Predict what happens with `linreg.fit(X2, stock, lr=0.1, epochs=100)`. Then run it.
2. Find a learning rate that doesn't blow up. How many epochs until the market weight gets anywhere near the beta you found in section 8?
3. Now **standardise** each feature: subtract its mean and divide by its standard deviation (lesson 00, section 2), so every column has mean 0 and standard deviation 1. Fit again with `lr=0.1`. What changed, and why? Think about the bowl from section 4: what shape is it when one knob is millions of times more sensitive than the other?
4. After standardising, the market weight means "per standard deviation of market return" rather than "per 1%". How would you convert it back into a beta?

In [ ]:
# 🔮 Prediction:

# Your turn: unscaled features


In [ ]:
# Your turn: standardised features


### What's going on (read after you've tried it)

With unscaled features, the bowl turns into an extremely long, narrow valley. Nudging the volume weight even slightly changes the loss enormously (it gets multiplied by millions), while the market weight barely matters by comparison. A learning rate small enough not to blow up in the steep direction is far too small to make any progress in the shallow one.

Standardising makes the bowl round again, so one learning rate suits every direction. That's why almost every real ML pipeline scales its features.

## 10. 🏭 From scratch to industry

### What the libraries do behind the scenes

Everything you built has a direct equivalent in the libraries you'll use at work. Now you know what's under the hood:

| What you built | scikit-learn | PyTorch |
|---|---|---|
| `predict` | `model.predict(X)` | `model(X)` |
| `mse` | `mean_squared_error(y, y_pred)` | `nn.MSELoss()` |
| `gradients` | hidden inside `.fit()` | `loss.backward()`, computed automatically (lesson 08) |
| `w -= lr * dw` | hidden inside `.fit()` | `optimizer.step()` with `torch.optim.SGD(..., lr=0.1)` |
| `fit` with gradient descent | `SGDRegressor` | your own training loop |
| `fit_closed_form` | `LinearRegression` (solves least squares directly, like yours) | |
| standardising | `StandardScaler` | done in your data pipeline |

A PyTorch training loop is the same five lines as your `fit`: predict, compute the loss, compute the gradients, step, repeat.

### Where linear regression is used

- **Beta and hedging.** Exactly this lesson. A desk holding a stock can short `beta ×` its value in an index future to cancel out most of its market risk.
- **Factor models.** Funds explain returns with several inputs at once (market, company size, value vs growth, momentum). That's multiple linear regression, as in section 9.
- **Risk and pricing.** As a first, explainable model that a regulator or risk committee can follow line by line.
- **The baseline.** Every fancier model has to beat linear regression to justify itself. If a neural net can't, you ship the line.

### Pitfalls in real data

- **Beta changes over time.** Fitting on 2020 data and using the result in 2026 assumes nothing changed. Practitioners use rolling windows, for example re-fitting every month on the last year.
- **Outliers.** One crash day can move the estimate a lot, because MSE squares big errors (section 3).
- **Data leakage.** When you standardise, compute the mean and standard deviation from the *training* data only. Using the whole dataset quietly leaks future information into the past (lesson 03, and fintech-lab's backtesting experiment).
- **Correlation isn't causation.** A big weight means the input *moves with* the output, not that it *causes* it.

### Interview-style questions

Answer these in your own words in the README. If you can, you understand this lesson.
1. Why do we square the errors instead of just adding them up?
2. What happens if the learning rate is too high? Too low?
3. Why scale features before gradient descent?
4. Linear regression has an exact solution. Why would anyone use gradient descent for it?
5. How would you check that your gradient code is correct?
6. You estimated a beta of 1.3 from last year's data. Why might it be wrong next year?

## 11. Wrap-up

**Explain it to a friend.** In the README, explain gradient descent in five sentences, with no symbols and no code, to someone who has never programmed. If you get stuck on a sentence, that's the bit to revisit.

Then fill in the rest of this lesson's README (*In my own words*, *What broke*, the interview answers, *Questions I still have*), and add the new terms to `GLOSSARY.md`: model, parameter, loss function, mean squared error, loss landscape, gradient, gradient descent, learning rate, epoch, feature, standardisation.

## Stretch (pick at least one)

- **A real beta.** Download about two years of daily closing prices for a stock and an index (from [Stooq](https://stooq.com) or Yahoo Finance), turn them into daily % returns, and estimate the stock's beta. Compare with the beta Yahoo Finance publishes. Why might they differ? Look up the period and frequency they use.
- **Rolling beta.** Estimate beta on a sliding one-year window and plot how it changes over time.
- **Stochastic gradient descent.** Instead of all 250 days per step, use a random batch of 32. Plot the loss curve. Why is it noisy, and why does nearly all of deep learning do this anyway?
- **Outliers.** Add a crash day where the market falls 10% and the stock falls 40%. How far does the fitted beta move? Then try mean *absolute* error instead (you'll need its slope) and compare.